In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE, URL = '/content/code', 'github.com/quocthangtrann/computer_vision_ADAS.git'
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
if not os.path.exists(CODE):
    subprocess.run(f'git clone https://{URL} {CODE}', shell=True, check=True, env=env)
else:
    subprocess.run(f'git -C {CODE} pull', shell=True, check=True, env=env)
subprocess.run(f'rsync -a --exclude .git {CODE}/repo/ /content/drive/MyDrive/capstone_ai/repo/', shell=True, check=True)
try: import segmentation_models_pytorch
except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch'], check=True)
os.chdir('/content/drive/MyDrive/capstone_ai')
print(subprocess.run(f'git -C {CODE} log --oneline -1', shell=True, capture_output=True, text=True).stdout.strip())
print('mã mới nhất đã ở capstone_ai/repo/ (runtime/, config/, tests/ chạy thẳng từ /content/code) | thư mục làm việc:', os.getcwd())


In [ ]:
import os, subprocess, torch
os.chdir('/content/drive/MyDrive/capstone_ai')
print('tiến trình train đang chạy:', subprocess.run('pgrep -af "[t]rain_v3.py"', shell=True, capture_output=True, text=True).stdout.strip() or 'không')
subprocess.run('pkill -f "[t]rain_v3.py"', shell=True)
print('đã dừng tiến trình thừa (nếu có)')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ (đổi runtime sang GPU)')
print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout.strip())


In [ ]:
import os, sys, glob, json
os.chdir('/content/drive/MyDrive/capstone_ai')
# CẤU HÌNH: chỉ huấn luyện B (fsnet_m_p1s5, 640x320). Đặt tiêu chí quyết định TRƯỚC khi chạy (xem cell so sánh).
N_SHARDS = 4             # số shard 640x320 (cần đã dựng ở 10_data); 0 = chỉ bộ thí điểm
SEEDS = [0, 1, 2]
KD = True                # True: chưng cất từ giáo viên TEACHER (tag _kdT); False: chỉ gói chống quá khớp (tag _reg)
EPOCHS = 60
BUDGET_MIN = None        # số phút: không bắt đầu run mới sau mức này (phiên Colab có giới hạn); None = không giới hạn
W, H, ARCH = 640, 320, 'fsnet_m_p1s5'
TEACHER = 'hd640s0_mitb2_c4aug_s0'
BASE = 'hd640s1_p1s5_kdT_c4aug_s*'       # B hiện tại (3 seed), mốc so sánh
TAG = f'hd{W}s{N_SHARDS}_p1s5_' + ('kdT' if KD else 'reg')
REG = ['--ema', '0.999', '--wd', '0.01', '--wd_nobn', '1', '--ls', '0.05', '--cutout', '2']   # gói chống quá khớp, như B
KDA = ['--kd', '1.0', '--kd_T', '2.0', '--teacher', TEACHER]
def ready(W=640, H=320):
    # shard HD đã dựng xong (có đủ _x.npy, _y.npy, _ids.csv) trong data/v2/b9_chunks_{W}x{H}/
    out = []
    for p in glob.glob(f'data/v2/b9_chunks*{W}x{H}*/shard_*_ids.csv'):
        s, d = os.path.basename(p)[:-8], os.path.dirname(p)
        if os.path.exists(f'{d}/{s}_x.npy') and os.path.exists(f'{d}/{s}_y.npy'): out.append(s)
    return sorted(out)

sh = ready(W, H)
need = [f'shard_{i:03d}' for i in range(N_SHARDS)]
miss = [s for s in need if s not in sh]
if miss: raise SystemExit(f'thiếu shard {W}x{H}: {miss} (đang có {sh}). Chạy 10_data trước, hoặc giảm N_SHARDS.')
if KD and not (os.path.exists(f'runs/{TEACHER}/final_metrics.json') and os.path.exists(f'runs/{TEACHER}/last.pt')):
    raise SystemExit(f'KD = True cần giáo viên runs/{TEACHER}/ đã huấn luyện xong (last.pt, final_metrics.json)')
sys.path.insert(0, 'repo'); import models_custom as mc
net = mc.build_fsnet(ARCH, num_classes=4); bad = mc.contract_violations(net)
assert not bad, f'{ARCH} vi phạm hợp đồng toán tử: {bad}'
print(f'{ARCH} {W}x{H}: {sum(r["mac"] for r in mc.layer_table(net, H, W)) / 1e9:.3f} GMAC | hợp đồng toán tử: đạt')
EXTRA = ['--arch', ARCH, '--w', str(W), '--h', str(H)] + (['--b9', ','.join(need)] if N_SHARDS else []) + REG + (KDA if KD else [])
print('run:', [f'{TAG}_c4aug_s{s}' for s in SEEDS], '| mốc so sánh:', BASE)
print('tham số train_v3.py:', ' '.join(EXTRA))


In [ ]:
import os, re, sys, time, subprocess
import torch; assert torch.cuda.is_available(), 'Chưa có GPU: đổi runtime sang GPU'
os.chdir('/content/drive/MyDrive/capstone_ai'); os.makedirs('runs', exist_ok=True)
T0 = time.time()
for s in SEEDS:
    n = f'{TAG}_c4aug_s{s}'
    if os.path.exists(f'runs/{n}/final_metrics.json'): print(n, 'đã xong, bỏ qua'); continue
    if BUDGET_MIN and (time.time() - T0) / 60 > BUDGET_MIN: print(f'quá {BUDGET_MIN} phút: không bắt đầu {n}, chạy lại cell sau'); break
    resume = os.path.exists(f'runs/{n}/last.pt')     # train_v3.py tự tiếp tục từ last.pt (cùng tên run)
    print(f'\n>>> {n}' + (' (tiếp tục từ last.pt)' if resume else '') + f'  [{(time.time() - T0) / 60:.0f} phút]', flush=True)
    cmd = [sys.executable, '-u', 'repo/train_v3.py', '--name', n, '--seed', str(s), '--epochs', str(EPOCHS)] + EXTRA
    pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'runs/{n}.log', 'a') as lg:
        for ln in pr.stdout:
            lg.write(ln); m = re.match(r'ep (\d+)/', ln)
            if not m or int(m.group(1)) in (1, 2) or int(m.group(1)) % 10 == 0: print(ln, end='', flush=True)
    rc = pr.wait(); print('mã thoát:', rc, flush=True)
    if rc != 0: print(f'LỖI ở {n}: dừng, dán đoạn log trên cho tôi'); break
    if not os.path.exists(f'runs/{n}/final_metrics.json'): print(f'LỖI ở {n}: mã thoát 0 nhưng không có final_metrics.json, dán log cho tôi'); break
else: print(f'\nXONG sau {(time.time() - T0) / 60:.0f} phút')


In [ ]:
import os, json, glob, numpy as np
os.chdir('/content/drive/MyDrive/capstone_ai')
# Tiêu chí đặt trước (CLAUDE.md): nhận mô hình mới thay B chỉ khi làn đang đi tăng >= 0,01 so với B với t >= 3 (Welch),
# IoU đường không giảm, false_floor không tăng (so trung bình), và TV2 xác nhận ngân sách. Số ở epoch cuối, lưới 320x160 (iou_320).
def load(pat): return [json.load(open(p)) for p in sorted(glob.glob(f'runs/{pat}/final_metrics.json'))]
def met(f):
    k = '_320' if 'iou_320' in f else ''
    i = f['iou' + k]
    return dict(ego=i[2], adj=i[3], notours=i[1], road=f['road_iou' + k], ff=f['false_floor' + k])
def welch(a, b):
    d = b.mean() - a.mean()
    return d, (d / np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b)) if len(a) > 1 and len(b) > 1 else float('nan'))
NEW = f'{TAG}_c4aug_s*'
A, B = load(BASE), load(NEW)
print(f'mốc B hiện tại {BASE}: n = {len(A)} | nhóm mới {NEW}: n = {len(B)}')
if NEW == BASE: print('nhóm mới trùng mốc (N_SHARDS = 1, KD = True): không có gì để so')
elif not A or not B: print('chưa đủ run để so')
else:
    r = {}
    for c, nm in [('ego', 'làn đang đi'), ('adj', 'làn kề'), ('notours', 'không thuộc ta'), ('road', 'IoU đường'), ('ff', 'false_floor (thấp là tốt)')]:
        a, b = np.array([met(f)[c] for f in A]), np.array([met(f)[c] for f in B])
        d, t = welch(a, b); r[c] = (d, t)
        sa = f'± {a.std(ddof=1):.4f}' if len(a) > 1 else '(n=1)'; sb = f'± {b.std(ddof=1):.4f}' if len(b) > 1 else '(n=1)'
        print(f'  {nm:26s} B {a.mean():.4f} {sa} | mới {b.mean():.4f} {sb} | hiệu {d:+.4f} (t = {t:+.1f})')
    ok_ego = r['ego'][0] >= 0.01 and r['ego'][1] >= 3
    ok_road, ok_ff = r['road'][0] >= 0, r['ff'][0] <= 0
    print(f'\ntiêu chí đặt trước: làn đang đi +0,01 và t >= 3: {"ĐẠT" if ok_ego else "CHƯA"} | IoU đường không giảm: {"ĐẠT" if ok_road else "CHƯA"} | '
          f'false_floor không tăng: {"ĐẠT" if ok_ff else "CHƯA"} | TV2 xác nhận ngân sách: (kiểm tay)')
    print('=> ' + ('ĐẠT phần đo được: đóng băng (cell dưới), lượng tử (40_int8) rồi đánh giá runtime (30_eval) trước khi thay B'
                   if ok_ego and ok_road and ok_ff else 'giữ B hiện tại'))
print('\nkhoảng cách train - test ở làn đang đi (ảnh train nhãn đủ, không aug; lưới của run):')
for nm, fs in [('B', A), ('mới', B)]:
    if fs and all('iou_trainsub' in f for f in fs):
        tr = np.array([f['iou_trainsub'][2] for f in fs]); te = np.array([f['iou'][2] for f in fs])
        print(f'  {nm:4s} train {tr.mean():.4f} - test {te.mean():.4f} = {tr.mean() - te.mean():+.4f} (n = {len(fs)})')


In [ ]:
import os, json, glob, shutil, hashlib, zipfile, numpy as np, torch
os.chdir('/content/drive/MyDrive/capstone_ai')
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
os.makedirs('models_final', exist_ok=True); os.makedirs('results', exist_ok=True)
# 1) đóng băng: chỉ lấy last.pt["model"] (trọng số đánh giá và triển khai, bản EMA nếu có). Tên đã có thì KHÔNG ghi đè.
FREEZE = {}     # ví dụ {'B2_hd640s4_p1s5_kdT_s0': 'runs/hd640s4_p1s5_kdT_c4aug_s0'}; để trống = chỉ làm mới bảng và context_pack
info = json.load(open('models_final/MODELS.json')) if os.path.exists('models_final/MODELS.json') else {}
for name, d in FREEZE.items():
    out = f'models_final/{name}_weights.pt'
    if name in info or os.path.exists(out): print(f'{name}: đã có, không ghi đè (đổi tên nếu muốn đóng băng bản khác)'); continue
    if not os.path.exists(f'{d}/final_metrics.json'): print(f'{name}: {d} chưa huấn luyện xong (thiếu final_metrics.json), bỏ qua'); continue
    ck = torch.load(f'{d}/last.pt', map_location='cpu', weights_only=False)
    torch.save(ck['model'], out)
    for f in ['config.json', 'final_metrics.json']: shutil.copy(f'{d}/{f}', f'models_final/{name}_{f}')
    info[name] = dict(run=d, epoch=int(ck.get('epoch', -1)), bytes=os.path.getsize(out), sha256=sha(out))
    print(f'{name}: epoch {info[name]["epoch"]} | {info[name]["bytes"] / 1e6:.1f} MB | sha256 {info[name]["sha256"][:16]}...')
json.dump(info, open('models_final/MODELS.json', 'w'), indent=1)
print('models_final/MODELS.json:', list(info))

# 2) bảng kết quả (epoch cuối, lưới 320x160, 408 ảnh test)
def L(pat):
    r = []
    for p in sorted(glob.glob(f'runs/{pat}/final_metrics.json')):
        f = json.load(open(p)); i = f.get('iou_320', f['iou']); r.append((i[2], i[3], i[1], f.get('road_iou_320', f['road_iou'])))
    return np.array(r)
G = [('Mốc thí điểm 2.826 ảnh, aug mạnh, 320x160', 'pilot_c4aug_s*'), ('+1 shard, 320x160', 'b9s1_c4aug_s*'),
     ('A: +4 shard, 320x160 (dự phòng, ngừng phát triển)', 'b9s4_c4aug_s*'), ('HD 640x320 p1s5_reg, thí điểm', 'hd640s0_p1s5_reg_c4aug_s*'),
     ('Teacher MiT-B2, HD, thí điểm', 'hd640s0_mitb2_c4aug_s0'), ('HD 640x320 p1s5_reg, +1 shard', 'hd640s1_p1s5_reg_c4aug_s*'),
     ('B: HD 640x320 p1s5 + chưng cất, +1 shard (mô hình chính)', 'hd640s1_p1s5_kdT_c4aug_s*')]
for p in sorted({os.path.basename(x)[:-len('_c4aug_s0')] for x in glob.glob('runs/hd640s*_p1s5_*_c4aug_s*') if os.path.basename(x)[-1].isdigit()}):
    n = p.split('_')[0][len('hd640s'):]
    if p not in ('hd640s0_p1s5_reg', 'hd640s1_p1s5_reg', 'hd640s1_p1s5_kdT') and n.isdigit():
        G.append((f'HD 640x320 {p[len("hd640s") + len(n) + 1:]}, +{n} shard', f'{p}_c4aug_s*'))
lines, res = ['| Nhóm | n | Làn đang đi | Làn kề | Không thuộc ta | IoU đường |', '|---|---|---|---|---|---|'], {}
for title, pat in G:
    a = L(pat)
    if not len(a): continue
    cells = [f'{a[:, j].mean():.4f}' + (f' ± {a[:, j].std(ddof=1):.4f}' if len(a) > 1 else '') for j in range(4)]
    lines.append(f'| {title} | {len(a)} | ' + ' | '.join(cells) + ' |'); res[title] = dict(n=len(a), mean=a.mean(0).tolist(), pattern=pat)
md = '# Kết quả huấn luyện (epoch cuối, lưới 320x160, 408 ảnh test, GPU L4)\n\n' + '\n'.join(lines) + '\n'
open('results/final_results.md', 'w', encoding='utf-8').write(md); json.dump(res, open('results/final_results.json', 'w'), indent=1, ensure_ascii=False)
print('\n' + md)

# 3) làm mới gói ngữ cảnh cho Claude Code (chỉ tệp chữ nhỏ)
HB = 'handoff_s3b_fsnet_m_p1s5_640x320'
pats = ['runs/*/config.json', 'runs/*/final_metrics.json', 'runs/*/diag_last.json', 'runs/*/metrics.csv', 'runs/summary.csv', 'eval_report/*',
        'data_spec_v1.json', 'data/cache_v1/cache_spec.json', 'data/v2/manifest_v2.csv', 'data/v2/split_report.json', 'data/v2/pilot_drv_manifest.csv',
        'handoff_s3a_v2/README.md', 'handoff_s3a_v2/preprocess_spec.json', 'handoff_s3a_v2/layer_table.csv', 'handoff_s3b/README.md',
        'handoff_s3b/int8_meta.json', 'handoff_s3b/int8_sim_summary.json', 'handoff_s3b/export_info.json', 'handoff_s3b/layer_table.csv',
        'handoff_s3b/int8_layers_pct.csv'] + [f'{HB}/{f}' for f in ['README.md', 'int8_meta.json', 'int8_sim_summary.json', 'export_info.json',
        'layer_table.csv', 'int8_layers_pct.csv', 'int8_layers_max.csv', 'preprocess_spec.json', 'MANIFEST.json']] + \
       ['eval_runtime/*.json', 'results/*', 'models_final/MODELS.json', 'models_final/*_final_metrics.json', 'models_final/*_config.json']
shutil.rmtree('context_pack', ignore_errors=True); os.makedirs('context_pack'); n = 0
for pat in pats:
    for p in glob.glob(pat):
        if os.path.isfile(p) and os.path.getsize(p) < 3_000_000:
            dst = os.path.join('context_pack', p); os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(p, dst); n += 1
with zipfile.ZipFile('context_pack.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for d, _, fs in os.walk('context_pack'):
        for f in fs: z.write(os.path.join(d, f), os.path.relpath(os.path.join(d, f), 'context_pack'))
print(n, 'tệp | context_pack.zip', round(os.path.getsize('context_pack.zip') / 1e6, 1), 'MB')
